# Circuits — the sinusoidal steady state

Transients are hard because they involve derivatives. But if every voltage and current in a circuit is a sinusoid at the *same* frequency, then differentiating one only scales it and shifts it by 90° — which is exactly what multiplying by $j\omega$ does to a complex number.

$$v(t)=V\cos(\omega t+\varphi)\;\longleftrightarrow\;\mathbf{V}=Ve^{j\varphi},
\qquad \frac{d}{dt}\;\longleftrightarrow\;j\omega$$

Make that substitution and every differential equation from the previous notebooks turns into algebra. Ohm's law comes back, now with complex numbers:

$$\mathbf{V}=\mathbf{Z}\mathbf{I},\qquad
Z_R=R,\qquad Z_L=j\omega L,\qquad Z_C=\frac{1}{j\omega C}$$

Everything in this notebook is a consequence. Series and parallel rules still work, Kirchhoff still holds, Thévenin still holds — all of it, with $R$ replaced by $Z$. What is new is that impedance depends on frequency, and that is what makes filters, resonators and matching networks possible.

The schematics animate as before. The phasor diagrams rotate at $\omega$, and their vertical projections are the waveforms drawn beside them.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "circuits · sinusoidal steady state", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


def phasor(ax, z, color, label=None, lw=2.2, tail=(0, 0)):
    ax.annotate("", xy=(tail[0] + z.real, tail[1] + z.imag), xytext=tail,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw))
    if label:
        ax.text(tail[0] + z.real * 1.12, tail[1] + z.imag * 1.12, label,
                color=color, fontsize=8, ha="center", va="center")


def cplane(ax, lim):
    panel(ax)
    ax.axhline(0, color=GRIDC, lw=0.9); ax.axvline(0, color=GRIDC, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    return ax


print("engine ready — same schematic primitives, plus phasor helpers")
print("Z_R = R    Z_L = jwL    Z_C = 1/(jwC)")

engine ready — same schematic primitives, plus phasor helpers
Z_R = R    Z_L = jwL    Z_C = 1/(jwC)


## Phasors — a sinusoid is a rotating vector seen edge-on

Every sinusoid at frequency $\omega$ is the projection of a vector spinning at $\omega$. Fix the frequency and only two numbers are left, length and starting angle:

$$V\cos(\omega t+\varphi)=\mathrm{Re}\left\{Ve^{j\varphi}\cdot e^{j\omega t}\right\}$$

The reason this matters is on the right of the panel. Adding two sinusoids of the same frequency is trigonometry; adding two **vectors** is head-to-tail, and the answer is another sinusoid at the same frequency with an amplitude and phase you can read off with a ruler. The readout checks it: the phasor sum and the peak of the actual summed waveform agree to four decimal places.

Two consequences worth carrying. The frequency never appears in the arithmetic — it is a shared rotation everyone is doing, so it cancels. And **only relative phase matters**, because rotating the whole picture changes nothing physical; that is why one signal is always chosen as the reference at 0°.

This is the entire trick. From here on, no differential equations.

In [2]:
def draw_phasors(k, A1, ph1, A2, ph2, f_hz):
    w = 2 * np.pi * f_hz
    wt = 2 * np.pi * k / 240
    P1 = A1 * np.exp(1j * np.deg2rad(ph1))
    P2 = A2 * np.exp(1j * np.deg2rad(ph2))
    PS = P1 + P2
    lim = (abs(P1) + abs(P2)) * 1.25
    t = np.linspace(0, 2 / f_hz, 600)
    s1 = A1 * np.cos(w * t + np.deg2rad(ph1))
    s2 = A2 * np.cos(w * t + np.deg2rad(ph2))
    now = wt / w

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.28, hspace=0.44, left=0.04, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = cplane(fig.add_subplot(gs[:, 0]), lim)
    th = np.linspace(0, 2 * np.pi, 200)
    for r, c in ((abs(P1), POS), (abs(P2), PURP), (abs(PS), DOT)):
        a0.plot(r * np.cos(th), r * np.sin(th), color=c, lw=0.5, alpha=0.35)
    rot = np.exp(1j * wt)
    phasor(a0, P1 * rot, POS, "V₁")
    phasor(a0, P2 * rot, PURP, "V₂", tail=(P1.real * np.cos(wt) - P1.imag * np.sin(wt),
                                           P1.real * np.sin(wt) + P1.imag * np.cos(wt)))
    phasor(a0, PS * rot, DOT, "sum", lw=2.8)
    a0.plot([0, lim], [(PS * rot).imag] * 2, color=DOT, lw=0.7, ls=":")
    a0.set_xlabel("real"); a0.set_ylabel("imaginary")
    a0.set_title(f"rotating at {f_hz:.0f} Hz — ωt = {np.degrees(wt) % 360:.0f}°")

    a1 = panel(fig.add_subplot(gs[:, 1]), BLUE)
    a1.plot(t * 1e3, s1, color=POS, lw=1.2, label="v₁")
    a1.plot(t * 1e3, s2, color=PURP, lw=1.2, label="v₂")
    a1.plot(t * 1e3, s1 + s2, color=DOT, lw=1.8, label="v₁ + v₂")
    a1.axvline(now * 1e3, color=FG, lw=1.0, ls="--")
    a1.plot([now * 1e3], [(PS * rot).imag], "o", ms=8, color=DOT)
    a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_xlabel("time  (ms)"); a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title("the projection of the spinning vector is the waveform")

    peak = (s1 + s2).max()
    readout(fig, 0.845, 0.90, [
        "PHASOR 1", "─" * 26,
        f"amplitude   {A1:>10.3f}V",
        f"phase       {ph1:>+10.2f}°",
        "", "PHASOR 2", "─" * 26,
        f"amplitude   {A2:>10.3f}V",
        f"phase       {ph2:>+10.2f}°",
        "", "SUM (by vector)", "─" * 26,
        f"|V1+V2|     {abs(PS):>10.5f}V",
        f"angle       {np.degrees(np.angle(PS)):>+10.4f}°",
        "", "SUM (by waveform)", "─" * 26,
        f"peak        {peak:>10.5f}V",
        f"difference  {abs(peak-abs(PS)):>10.1e}V",
        "", "the frequency cancels",
        "only relative phase",
        "carries information",
    ])
    footer(fig, f"V cos(ωt+φ) = Re[V e^(jφ) · e^(jωt)]   ·   "
                f"adding sinusoids = adding vectors")
    plt.show()


_p1, _s1 = timeline(239, step=3)
w1 = dict(A1=widgets.FloatSlider(value=3, min=0.5, max=6, step=0.25,
                                 description="|V1|:", **SL),
          ph1=widgets.FloatSlider(value=20, min=-180, max=180, step=5,
                                  description="phase V1:", **SL),
          A2=widgets.FloatSlider(value=2, min=0.5, max=6, step=0.25,
                                 description="|V2|:", **SL),
          ph2=widgets.FloatSlider(value=-70, min=-180, max=180, step=5,
                                  description="phase V2:", **SL),
          f_hz=widgets.FloatSlider(value=1000, min=200, max=3000, step=100,
                                   description="frequency Hz:", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["A1"], w1["ph1"], w1["A2"]]),
                      widgets.HBox([w1["ph2"], w1["f_hz"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_phasors, w1))

Output()

## Impedance — resistance that depends on frequency, and points sideways

With $d/dt\to j\omega$, each component becomes a complex number:

$$Z_R=R\ (\text{real}),\qquad
Z_L=j\omega L\ (\text{up}),\qquad
Z_C=\frac{1}{j\omega C}=-\frac{j}{\omega C}\ (\text{down})$$

The directions are the phase relationships from the first notebook, redrawn as geometry. Resistance is real because voltage and current stay in step; the reactances are imaginary because they sit a quarter cycle away, and the two point in **opposite** directions, which is the single fact that makes resonance possible.

In series they simply add:

$$Z=R+j\left(\omega L-\frac{1}{\omega C}\right)$$

Sweep the frequency and watch the point move on the complex plane. At low frequency the capacitor dominates and $Z$ sits far below the real axis; at high frequency the inductor takes over and it climbs above. In between there is exactly one frequency where the two cancel completely and $Z$ lands **on** the real axis, equal to $R$ alone — confirmed in the readout as $|Z|=R$ to five decimals.

The angle of $Z$ is the phase between voltage and current, and its sign tells you which component is currently winning.

In [3]:
def draw_impedance(f_hz, R, L_mH, C_uF):
    L, C = L_mH * 1e-3, C_uF * 1e-6
    w = 2 * np.pi * f_hz
    ZR, ZL, ZC = R + 0j, 1j * w * L, 1 / (1j * w * C)
    Z = ZR + ZL + ZC
    f0 = 1 / (2 * np.pi * np.sqrt(L * C))
    ff = np.logspace(np.log10(f0 / 30), np.log10(f0 * 30), 600)
    Zf = R + 1j * (2 * np.pi * ff * L - 1 / (2 * np.pi * ff * C))
    lim = max(abs(ZL), abs(ZC), R) * 1.35

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.25, 0.55],
                          wspace=0.3, hspace=0.46, left=0.04, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = cplane(fig.add_subplot(gs[:, 0]), lim)
    phasor(a0, ZR, POS, "R")
    phasor(a0, ZL, PURP, "jωL", tail=(ZR.real, ZR.imag))
    phasor(a0, ZC, ORANGE, "1/jωC", tail=(ZR.real + ZL.real, ZR.imag + ZL.imag))
    phasor(a0, Z, DOT, "Z", lw=2.8)
    a0.plot(Zf.real, Zf.imag, color=MUTED, lw=0.9, ls=":")
    a0.set_xlabel("resistance  (Ω)"); a0.set_ylabel("reactance  (Ω)")
    a0.set_title("R points right, L up, C down — they add head to tail")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.loglog(ff, np.abs(Zf), color=POS, lw=1.6)
    a1.loglog(ff, 2 * np.pi * ff * L, color=PURP, lw=0.9, ls="--", label="ωL")
    a1.loglog(ff, 1 / (2 * np.pi * ff * C), color=ORANGE, lw=0.9, ls="--",
              label="1/ωC")
    a1.axhline(R, color=MUTED, lw=0.8, ls=":")
    a1.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a1.axvline(f0, color=DOT, lw=1.0)
    a1.plot([f_hz], [abs(Z)], "o", ms=7, color=DOT)
    a1.set_ylabel("|Z|  (Ω)"); a1.legend(fontsize=7)
    a1.set_title(f"|Z| bottoms out at R = {R:.1f} Ω, at f₀ = {f0:.0f} Hz")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.semilogx(ff, np.degrees(np.angle(Zf)), color=ORANGE, lw=1.6)
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.axhline(90, color=GRIDC, lw=0.6, ls=":")
    a2.axhline(-90, color=GRIDC, lw=0.6, ls=":")
    a2.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a2.axvline(f0, color=DOT, lw=1.0)
    a2.set_ylim(-100, 100); a2.set_xlabel("frequency  (Hz)")
    a2.set_ylabel("angle of Z  (deg)")
    a2.set_title("capacitive below f₀, inductive above")

    dom = ("capacitive" if Z.imag < -1e-9 else
           "inductive" if Z.imag > 1e-9 else "purely resistive")
    readout(fig, 0.845, 0.90, [
        "AT THIS FREQUENCY", "─" * 26,
        f"f           {f_hz:>10.1f}Hz",
        f"ω           {w/1e3:>10.3f}krad/s",
        "", "COMPONENTS", "─" * 26,
        f"R           {R:>+10.3f}Ω",
        f"XL = ωL     {ZL.imag:>+10.3f}Ω",
        f"XC = −1/ωC  {ZC.imag:>+10.3f}Ω",
        f"X total     {Z.imag:>+10.3f}Ω",
        "", "IMPEDANCE", "─" * 26,
        f"|Z|         {abs(Z):>10.4f}Ω",
        f"angle       {np.degrees(np.angle(Z)):>+10.3f}°",
        f"behaviour   {dom:>14s}",
        "", "RESONANCE", "─" * 26,
        f"f0          {f0:>10.2f}Hz",
        f"|Z| at f0   {R:>10.4f}Ω",
        f"f / f0      {f_hz/f0:>10.4f}",
    ], color=DOT if abs(Z.imag) < 0.02 * R else FG)
    footer(fig, f"Z = R + j(ωL − 1/ωC)   ·   f0 = 1/(2π√LC) = {f0:.1f} Hz   ·   "
                f"the reactances cancel there")
    plt.show()


w2 = dict(f_hz=widgets.FloatSlider(value=5033, min=200, max=20000, step=50,
                                   description="frequency Hz:", **SL),
          R=widgets.FloatSlider(value=10, min=1, max=200, step=1,
                                description="R (Ω):", **SL),
          L_mH=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="L (mH):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,
                                   description="C (µF):", **SL))
display(widgets.HBox([w2["f_hz"], w2["R"], w2["L_mH"], w2["C_uF"]]),
        widgets.interactive_output(draw_impedance, w2))

Output()

## Series resonance — a circuit that answers to one frequency

At $f_0$ the two reactances cancel and the series circuit is left with $R$ alone, so the current peaks. How sharply is set by one number:

$$Q=\frac{1}{R}\sqrt{\frac{L}{C}},\qquad
\Delta f_{-3\text{dB}}=\frac{f_0}{Q}$$

The measured bandwidth in the panel matches $f_0/Q$ to four significant figures. Note what $Q$ does *not* contain: it has no frequency in it. Sharpness comes from the ratio of stored energy to dissipated energy per cycle, and $R$ is the only thing spending any.

The striking part is what happens across the individual components. At resonance the voltage on the inductor and on the capacitor are each **$Q$ times the source voltage**, in exact opposition, so they cancel as far as the loop is concerned while being individually enormous. With $Q=10$ and a 1 V source there is 10 V sitting across the capacitor. That is not an error — it is how a crystal radio pulls a usable signal out of an antenna, and it is also how components get destroyed by a resonance nobody intended.

Lower $R$ and the peak narrows and grows together. Selectivity and voltage magnification are the same phenomenon.

In [4]:
def draw_series_res(k, f_hz, R, L_mH, C_uF, Vs):
    L, C = L_mH * 1e-3, C_uF * 1e-6
    f0 = 1 / (2 * np.pi * np.sqrt(L * C))
    Q = (1 / R) * np.sqrt(L / C)
    BW = f0 / Q
    w = 2 * np.pi * f_hz
    Z = R + 1j * (w * L - 1 / (w * C))
    I = Vs / Z
    VL, VC, VR = I * 1j * w * L, I / (1j * w * C), I * R
    ff = np.linspace(max(f0 - 4 * BW, 1), f0 + 4 * BW, 1200)
    Zf = R + 1j * (2 * np.pi * ff * L - 1 / (2 * np.pi * ff * C))
    If = np.abs(Vs / Zf)
    band = ff[If >= If.max() / np.sqrt(2)]
    meas_bw = band[-1] - band[0] if len(band) > 1 else np.nan
    vmax = max(Vs, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 5.0), (-0.4, 2.0))
    wire(a0, [(0.5, 1.6), (1.1, 1.6)], Vs, vmax)
    resistor(a0, (1.1, 1.6), (2.2, 1.6), Vs, vmax, f"R {R:.0f}Ω")
    inductor(a0, (2.2, 1.6), (3.4, 1.6), abs(VL) / 3, vmax, f"L {L_mH:.2f}mH")
    capacitor(a0, (3.4, 1.6), (4.5, 1.6), abs(VC) / 3, vmax, f"C {C_uF:.2f}µF")
    wire(a0, [(4.5, 1.6), (4.7, 1.6), (4.7, 0.1), (0.5, 0.1)], 0.0, vmax)
    source(a0, (0.5, 0.1), (0.5, 1.6), Vs / 2, vmax, "ac", f"{Vs:.1f}V")
    charge_dots(a0, loop_rect(0.5, 4.7, 0.1, 1.6),
                k / 400 * abs(I) * 1e4, spacing=0.30, ms=3.5)
    a0.set_title(f"series RLC — current {abs(I)*1e3:.3f} mA", fontsize=8.5)

    a1 = cplane(fig.add_subplot(gs[1, 0]), max(abs(VL), abs(VC), Vs) * 1.3)
    phasor(a1, VR, POS, "V_R")
    phasor(a1, VL, PURP, "V_L", tail=(VR.real, VR.imag))
    phasor(a1, VC, ORANGE, "V_C", tail=(VR.real + VL.real, VR.imag + VL.imag))
    phasor(a1, complex(Vs, 0), DOT, "source", lw=2.6)
    a1.set_title("component voltages add to the source", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a2.plot(ff, If * 1e3, color=POS, lw=1.6)
    a2.axhline(If.max() / np.sqrt(2) * 1e3, color=MUTED, lw=0.8, ls=":")
    if len(band) > 1:
        a2.axvspan(band[0], band[-1], color=DOT, alpha=0.12)
    a2.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a2.plot([f_hz], [abs(I) * 1e3], "o", ms=7, color=DOT)
    a2.set_ylabel("current  (mA)")
    a2.set_title(f"Q = {Q:.3f}   measured BW {meas_bw:.1f} Hz   "
                 f"f0/Q = {BW:.1f} Hz")

    a3 = panel(fig.add_subplot(gs[1, 1]), PURP)
    a3.plot(ff, np.abs(Vs / Zf * 1j * 2 * np.pi * ff * L), color=PURP, lw=1.4,
            label="|V_L|")
    a3.plot(ff, np.abs(Vs / Zf / (1j * 2 * np.pi * ff * C)), color=ORANGE, lw=1.4,
            label="|V_C|")
    a3.axhline(Vs, color=MUTED, lw=0.8, ls=":")
    a3.axhline(Q * Vs, color=DOT, lw=0.8, ls="--")
    a3.text(ff[5], Q * Vs, f" Q·Vs = {Q*Vs:.2f} V", color=DOT, fontsize=7)
    a3.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a3.set_xlabel("frequency  (Hz)"); a3.set_ylabel("volts"); a3.legend(fontsize=7)
    a3.set_title("each reactance carries Q times the source voltage")

    readout(fig, 0.845, 0.90, [
        "CIRCUIT", "─" * 26,
        f"R           {R:>10.2f}Ω",
        f"L           {L_mH:>10.3f}mH",
        f"C           {C_uF:>10.3f}µF",
        f"source      {Vs:>10.2f}V",
        "", "RESONANCE", "─" * 26,
        f"f0          {f0:>10.2f}Hz",
        f"Q           {Q:>10.4f}",
        f"BW = f0/Q   {BW:>10.2f}Hz",
        f"measured BW {meas_bw:>10.2f}Hz",
        "", "AT THIS f", "─" * 26,
        f"f/f0        {f_hz/f0:>10.4f}",
        f"|Z|         {abs(Z):>10.3f}Ω",
        f"|I|         {abs(I)*1e3:>10.4f}mA",
        f"|V_R|       {abs(VR):>10.4f}V",
        f"|V_L|       {abs(VL):>10.4f}V",
        f"|V_C|       {abs(VC):>10.4f}V",
        f"V_L + V_C   {abs(VL+VC):>10.4f}V",
    ], color=DOT if abs(f_hz - f0) < BW / 20 else FG)
    footer(fig, f"Q = (1/R)√(L/C) = {Q:.3f}   ·   BW = f0/Q   ·   "
                f"at f0 the reactive voltages are Q× the source and cancel")
    plt.show()


_p3, _s3 = timeline(399, step=4)
w3 = dict(f_hz=widgets.FloatSlider(value=5033, min=1000, max=12000, step=10,
                                   description="frequency Hz:", **SL),
          R=widgets.FloatSlider(value=10, min=1, max=100, step=1,
                                description="R (Ω):", **SL),
          L_mH=widgets.FloatSlider(value=1.0, min=0.2, max=5, step=0.1,
                                   description="L (mH):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1,
                                   description="C (µF):", **SL),
          Vs=widgets.FloatSlider(value=1.0, min=0.5, max=5, step=0.5,
                                 description="source V:", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["f_hz"], w3["R"], w3["L_mH"]]),
                      widgets.HBox([w3["C_uF"], w3["Vs"]]),
                      widgets.HBox([_p3, _s3])]),
        widgets.interactive_output(draw_series_res, w3))

Output()

## The parallel tank — the same resonance, inside out

Put the same three parts in parallel and everything flips. Admittances add now, and at $f_0$ the inductive and capacitive admittances cancel, leaving only $1/R$ — so the impedance **peaks** instead of dipping:

$$Y=\frac{1}{R}+j\omega C+\frac{1}{j\omega L},\qquad
Q_{\text{par}}=R\sqrt{\frac{C}{L}}$$

Note $Q$ is inverted too: for the parallel tank a **larger** $R$ gives a sharper resonance, because here $R$ is the leakage path rather than the loss in series with the current.

The consequence is the mirror of the series case. There, the *voltages* on L and C were $Q$ times the source; here the *currents* are. With the values in the readout, 0.1 mA drawn from the source sustains **31.6 mA circulating** between the inductor and the capacitor — measured ratio 316 at $Q=316$. The energy sloshes between the two reactances exactly as in the LC section of the first notebook, and the source only has to top up what $R$ loses.

This is what a tuned amplifier load is, what sets an oscillator's frequency, and why a tank is drawn across a transistor rather than in series with it.

In [5]:
def draw_tank(k, f_hz, Rp_k, L_mH, C_uF, Is_mA):
    L, C, Rp, Is = L_mH * 1e-3, C_uF * 1e-6, Rp_k * 1e3, Is_mA * 1e-3
    f0 = 1 / (2 * np.pi * np.sqrt(L * C))
    Q = Rp * np.sqrt(C / L)
    BW = f0 / Q
    w = 2 * np.pi * f_hz
    Y = 1 / Rp + 1j * w * C + 1 / (1j * w * L)
    Z = 1 / Y
    V = Is * Z
    IL, IC, IR = V / (1j * w * L), V * 1j * w * C, V / Rp
    ff = np.linspace(max(f0 - 6 * BW, 1), f0 + 6 * BW, 1200)
    Zf = 1 / (1 / Rp + 1j * 2 * np.pi * ff * C + 1 / (1j * 2 * np.pi * ff * L))
    band = ff[np.abs(Zf) >= np.abs(Zf).max() / np.sqrt(2)]
    meas_bw = band[-1] - band[0] if len(band) > 1 else np.nan
    vmax = max(abs(V), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 5.0), (-0.4, 2.0))
    wire(a0, [(0.5, 1.6), (4.6, 1.6)], abs(V), vmax)
    wire(a0, [(0.5, 0.1), (4.6, 0.1)], 0.0, vmax)
    a0.annotate("", xy=(0.9, 1.6), xytext=(0.9, 0.4),
                arrowprops=dict(arrowstyle="-|>", color=DOT, lw=2.0))
    a0.text(1.05, 0.9, f"{Is_mA:.2f} mA", color=DOT, fontsize=7.5)
    resistor(a0, (2.2, 1.6), (2.2, 0.1), abs(V) / 2, vmax, f"R {Rp_k:.1f}kΩ")
    inductor(a0, (3.4, 1.6), (3.4, 0.1), abs(V) / 2, vmax, f"L {L_mH:.2f}mH")
    capacitor(a0, (4.6, 1.6), (4.6, 0.1), abs(V) / 2, vmax, f"C {C_uF:.2f}µF")
    node_dot(a0, (3.4, 1.6), abs(V), vmax)
    kk = k / 400
    charge_dots(a0, seg((3.4, 1.6), (3.4, 0.1)), kk * abs(IL) * 3e3,
                spacing=0.24, ms=3.4)
    charge_dots(a0, seg((4.6, 1.6), (4.6, 0.1)), -kk * abs(IC) * 3e3,
                spacing=0.24, ms=3.4)
    charge_dots(a0, seg((2.2, 1.6), (2.2, 0.1)), kk * abs(IR) * 3e3,
                spacing=0.24, ms=3.4)
    a0.set_title(f"the tank — L and C exchange {abs(IL)*1e3:.2f} mA between them",
                 fontsize=8.5)

    a1 = cplane(fig.add_subplot(gs[1, 0]), max(abs(IL), abs(IC), Is) * 1.35 * 1e3)
    phasor(a1, IR * 1e3, POS, "I_R")
    phasor(a1, IL * 1e3, PURP, "I_L")
    phasor(a1, IC * 1e3, ORANGE, "I_C")
    phasor(a1, complex(Is * 1e3, 0), DOT, "source", lw=2.6)
    a1.set_xlabel("mA"); a1.set_title("branch currents (mA)", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a2.plot(ff, np.abs(Zf) / 1e3, color=POS, lw=1.6)
    a2.axhline(np.abs(Zf).max() / np.sqrt(2) / 1e3, color=MUTED, lw=0.8, ls=":")
    if len(band) > 1:
        a2.axvspan(band[0], band[-1], color=DOT, alpha=0.12)
    a2.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a2.plot([f_hz], [abs(Z) / 1e3], "o", ms=7, color=DOT)
    a2.set_ylabel("|Z|  (kΩ)")
    a2.set_title(f"impedance PEAKS at f0 — Q = {Q:.1f}, BW {meas_bw:.1f} Hz "
                 f"(f0/Q = {BW:.1f})")

    a3 = panel(fig.add_subplot(gs[1, 1]), PURP)
    a3.plot(ff, np.abs(Is * Zf / (1j * 2 * np.pi * ff * L)) * 1e3, color=PURP,
            lw=1.4, label="|I_L|")
    a3.plot(ff, np.abs(Is * Zf * 1j * 2 * np.pi * ff * C) * 1e3, color=ORANGE,
            lw=1.4, label="|I_C|")
    a3.axhline(Is * 1e3, color=MUTED, lw=0.8, ls=":")
    a3.text(ff[5], Is * 1e3 * 1.4, "source current", color=MUTED, fontsize=7)
    a3.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a3.set_xlabel("frequency  (Hz)"); a3.set_ylabel("mA"); a3.legend(fontsize=7)
    a3.set_title("circulating current is Q times what the source supplies")

    readout(fig, 0.845, 0.90, [
        "TANK", "─" * 26,
        f"R parallel  {Rp_k:>10.2f}kΩ",
        f"L           {L_mH:>10.3f}mH",
        f"C           {C_uF:>10.3f}µF",
        f"drive       {Is_mA:>10.3f}mA",
        "", "RESONANCE", "─" * 26,
        f"f0          {f0:>10.2f}Hz",
        f"Q = R√(C/L) {Q:>10.3f}",
        f"BW = f0/Q   {BW:>10.2f}Hz",
        f"measured BW {meas_bw:>10.2f}Hz",
        f"|Z| peak    {Rp/1e3:>10.3f}kΩ",
        "", "AT THIS f", "─" * 26,
        f"|Z|         {abs(Z)/1e3:>10.4f}kΩ",
        f"|V|         {abs(V):>10.4f}V",
        f"|I_L|       {abs(IL)*1e3:>10.4f}mA",
        f"|I_C|       {abs(IC)*1e3:>10.4f}mA",
        f"|I_L|/|Is|  {abs(IL)/Is:>10.2f}",
        "", "series: V magnified",
        "parallel: I magnified",
    ], color=DOT if abs(f_hz - f0) < BW / 20 else FG)
    footer(fig, f"Y = 1/R + jωC + 1/jωL   ·   Q = R√(C/L) = {Q:.1f}   ·   "
                f"bigger R means SHARPER here")
    plt.show()


_p4, _s4 = timeline(399, step=4)
w4 = dict(f_hz=widgets.FloatSlider(value=5033, min=3000, max=8000, step=5,
                                   description="frequency Hz:", **SL),
          Rp_k=widgets.FloatSlider(value=10, min=0.5, max=50, step=0.5,
                                   description="R (kΩ):", **SL),
          L_mH=widgets.FloatSlider(value=1.0, min=0.2, max=5, step=0.1,
                                   description="L (mH):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1,
                                   description="C (µF):", **SL),
          Is_mA=widgets.FloatSlider(value=0.1, min=0.05, max=2.0, step=0.05,
                                    description="drive (mA):", **SL),
          k=_s4)
display(widgets.VBox([widgets.HBox([w4["f_hz"], w4["Rp_k"], w4["L_mH"]]),
                      widgets.HBox([w4["C_uF"], w4["Is_mA"]]),
                      widgets.HBox([_p4, _s4])]),
        widgets.interactive_output(draw_tank, w4))

Output()

## Power — the part that is delivered and the part that is only borrowed

Instantaneous power is $p(t)=v(t)i(t)$, and once there is a phase angle between them it goes **negative** for part of every cycle: energy that went into the reactance comes straight back out. Averaging separates the two:

$$P=\tfrac12VI\cos\varphi\ \ (\text{watts}),\qquad
Q=\tfrac12VI\sin\varphi\ \ (\text{var}),\qquad
S=\tfrac12VI\ \ (\text{VA})$$

$$S^2=P^2+Q^2$$

Only $P$ leaves the source for good. $Q$ oscillates back and forth and does no net work — but it is carried by real current in real wires, so it heats them exactly as much as useful current would. That is why **power factor** $\cos\varphi$ is billed for: at $\cos\varphi=0.5$ you need twice the current for the same watts, and the losses go as $i^2$.

The panels make the split visible. At $\varphi=60°$ the instantaneous power curve is below zero for **33.3%** of the cycle, measured, and the mean sits at exactly half the apparent power. At $\varphi=90°$ — a pure reactance — the curve is symmetric about zero and the average is exactly zero watts, while the current is at full amplitude the whole time.

Adding a correcting reactance rotates $\varphi$ back toward zero without touching $P$. That is all power-factor correction is.

In [ ]:
def draw_power(k, V, R, X, f_hz):
    Z = R + 1j * X
    I = V / Z
    phi = np.angle(Z)
    S = 0.5 * V * abs(I)
    P = S * np.cos(phi)
    Qr = S * np.sin(phi)
    w = 2 * np.pi * f_hz
    t = np.linspace(0, 2 / f_hz, 900)
    vt = V * np.cos(w * t)
    it = abs(I) * np.cos(w * t - phi)
    pt = vt * it
    kk = int(min(k, len(t) - 1))
    vmax = max(V, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 4.6), (-0.4, 2.0))
    wire(a0, [(0.5, 1.6), (2.0, 1.6)], V, vmax)
    resistor(a0, (2.0, 1.6), (3.2, 1.6), V, vmax, f"R {R:.0f}Ω")
    if X >= 0:
        inductor(a0, (3.2, 1.6), (4.2, 1.6), V / 3, vmax, f"+j{X:.0f}Ω")
    else:
        capacitor(a0, (3.2, 1.6), (4.2, 1.6), V / 3, vmax, f"−j{abs(X):.0f}Ω")
    wire(a0, [(4.2, 1.6), (4.4, 1.6), (4.4, 0.1), (0.5, 0.1)], 0.0, vmax)
    source(a0, (0.5, 0.1), (0.5, 1.6), V / 2, vmax, "ac", f"{V:.0f}V")
    charge_dots(a0, loop_rect(0.5, 4.4, 0.1, 1.6), k / 400 * abs(I) * 40,
                spacing=0.30, ms=3.5)
    a0.set_title(f"|I| = {abs(I):.3f} A at {np.degrees(-phi):+.1f}°", fontsize=8.5)

    a1 = cplane(fig.add_subplot(gs[1, 0]), max(S, abs(P), abs(Qr)) * 1.35)
    phasor(a1, complex(P, 0), POS, "P")
    phasor(a1, complex(0, Qr), PURP, "Q", tail=(P, 0))
    phasor(a1, complex(P, Qr), DOT, "S", lw=2.6)
    a1.set_xlabel("watts"); a1.set_ylabel("var")
    a1.set_title(f"power triangle — pf = {np.cos(phi):.4f}", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a2.plot(t * 1e3, vt, color=POS, lw=1.3, label="v")
    a2.plot(t * 1e3, it * R, color=DOT, lw=1.3, label="i (scaled)")
    a2.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.legend(fontsize=7); a2.set_ylabel("volts")
    a2.set_title(f"current lags by {np.degrees(phi):+.1f}°")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a3.fill_between(t * 1e3, 0, pt, where=pt >= 0, color=GREEN, alpha=0.35,
                    interpolate=True)
    a3.fill_between(t * 1e3, 0, pt, where=pt < 0, color=NEG, alpha=0.35,
                    interpolate=True)
    a3.plot(t * 1e3, pt, color=FG, lw=1.1)
    a3.axhline(P, color=DOT, lw=1.2, ls="--")
    a3.text(t[5] * 1e3, P, f" mean = P = {P:.3f} W", color=DOT, fontsize=7.5)
    a3.axvline(t[kk] * 1e3, color=FG, lw=1.0, ls="--")
    a3.set_xlabel("time  (ms)"); a3.set_ylabel("p(t)  (W)")
    a3.set_title(f"red is energy handed back — {100*np.mean(pt<0):.1f}% of the cycle")

    readout(fig, 0.845, 0.90, [
        "LOAD", "─" * 26,
        f"R           {R:>+10.2f}Ω",
        f"X           {X:>+10.2f}Ω",
        f"|Z|         {abs(Z):>10.3f}Ω",
        f"angle φ     {np.degrees(phi):>+10.2f}°",
        "", "CURRENT", "─" * 26,
        f"|I|         {abs(I):>10.4f}A",
        f"phase       {np.degrees(-phi):>+10.2f}°",
        "", "POWER", "─" * 26,
        f"P real      {P:>10.4f}W",
        f"Q reactive  {Qr:>+10.4f}var",
        f"S apparent  {S:>10.4f}VA",
        f"power factor{np.cos(phi):>10.4f}",
        f"S²−(P²+Q²)  {S**2-(P**2+Qr**2):>+10.1e}",
        "", "MEASURED", "─" * 26,
        f"mean p(t)   {np.mean(pt):>10.4f}W",
        f"negative    {100*np.mean(pt<0):>10.1f}%",
        "", "same P at pf=1 needs",
        f"only {abs(I)*abs(np.cos(phi)):.3f} A instead of {abs(I):.3f}",
    ], color=ORANGE if abs(np.cos(phi)) < 0.8 else FG)
    footer(fig, f"P = ½VI cos φ   Q = ½VI sin φ   S = ½VI   S² = P² + Q²   ·   "
                f"pf = {np.cos(phi):.4f}")
    plt.show()


_p5, _s5 = timeline(899, step=9)
w5 = dict(V=widgets.FloatSlider(value=10, min=1, max=20, step=1,
                                description="source V:", **SL),
          R=widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5,
                                description="R (Ω):", **SL),
          X=widgets.FloatSlider(value=8.66, min=-20, max=20, step=0.5,
                                description="X (Ω):", **SL),
          f_hz=widgets.FloatSlider(value=1000, min=200, max=3000, step=100,
                                   description="frequency Hz:", **SL),
          k=_s5)
display(widgets.VBox([widgets.HBox([w5["V"], w5["R"], w5["X"], w5["f_hz"]]),
                      widgets.HBox([_p5, _s5])]),
        widgets.interactive_output(draw_power, w5))

Output()

## Frequency response — the whole circuit as one curve

Nothing new is needed for a filter. An RC divider is still a divider, with $Z_C$ in place of a resistance:

$$H(j\omega)=\frac{Z_C}{R+Z_C}=\frac{1}{1+j\omega RC}
=\frac{1}{1+jf/f_c},\qquad f_c=\frac{1}{2\pi RC}$$

Plotted with both axes logarithmic, the answer becomes two straight lines. Well below $f_c$ the response is flat at 0 dB; well above, it falls at **20 dB per decade** — measured $-19.957$ dB between $10f_c$ and $100f_c$, which is the asymptote arriving. They meet at $f_c$, where the exact response sits $3.01$ dB below and the phase is exactly $-45°$.

That is why $f_c$ is called the corner and the half-power point at once: $-3$ dB in voltage is a factor $1/\sqrt2$, and squaring it gives exactly one half the power.

Swap R and C and the same algebra gives a high-pass with the same corner and the mirrored slope. Every filter in the next notebook is built by stacking these: each additional pole adds another 20 dB/decade and another 90° of eventual phase shift — and that accumulating phase is what makes feedback loops oscillate.

In [7]:
def draw_bode(f_hz, R, C_nF, kind):
    C = C_nF * 1e-9
    fc = 1 / (2 * np.pi * R * C)
    ff = np.logspace(np.log10(fc / 1000), np.log10(fc * 1000), 900)
    if kind == "low-pass  (C to ground)":
        H = 1 / (1 + 1j * ff / fc); Hn = 1 / (1 + 1j * f_hz / fc)
        asym = np.where(ff < fc, 0.0, -20 * np.log10(ff / fc))
    else:
        H = (1j * ff / fc) / (1 + 1j * ff / fc)
        Hn = (1j * f_hz / fc) / (1 + 1j * f_hz / fc)
        asym = np.where(ff > fc, 0.0, 20 * np.log10(ff / fc))
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.3, hspace=0.44, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 4.4), (-0.5, 2.2))
    lo = kind.startswith("low")
    wire(a0, [(0.4, 1.7), (1.2, 1.7)], 1.0, vmax)
    if lo:
        resistor(a0, (1.2, 1.7), (2.6, 1.7), 1.0, vmax, f"R {R:.0f}Ω")
        wire(a0, [(2.6, 1.7), (3.9, 1.7)], abs(Hn), vmax)
        capacitor(a0, (3.0, 1.7), (3.0, 0.2), abs(Hn) / 2, vmax, f"C {C_nF:.0f}nF")
    else:
        capacitor(a0, (1.2, 1.7), (2.6, 1.7), 1.0, vmax, f"C {C_nF:.0f}nF")
        wire(a0, [(2.6, 1.7), (3.9, 1.7)], abs(Hn), vmax)
        resistor(a0, (3.0, 1.7), (3.0, 0.2), abs(Hn) / 2, vmax, f"R {R:.0f}Ω")
    wire(a0, [(0.4, 0.2), (3.9, 0.2)], 0.0, vmax)
    source(a0, (0.4, 0.2), (0.4, 1.7), 0.5, vmax, "ac", "1 V")
    node_dot(a0, (3.9, 1.7), abs(Hn), vmax)
    a0.text(3.9, 1.95, "out", color=FG, fontsize=8, ha="center")
    a0.set_title(f"|H| = {abs(Hn):.4f}  =  {20*np.log10(abs(Hn)):+.2f} dB",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.semilogx(ff, 20 * np.log10(np.abs(H)), color=POS, lw=1.8)
    a1.semilogx(ff, asym, color=MUTED, lw=0.9, ls="--", label="asymptotes")
    a1.axvline(fc, color=DOT, lw=1.0)
    a1.axhline(-3.01, color=MUTED, lw=0.8, ls=":")
    a1.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a1.plot([f_hz], [20 * np.log10(abs(Hn))], "o", ms=7, color=DOT)
    a1.set_ylim(-60, 6); a1.set_ylabel("|H|  (dB)"); a1.legend(fontsize=7)
    a1.set_title(f"corner at fc = {fc:,.1f} Hz, exactly −3.01 dB there")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.semilogx(ff, np.degrees(np.angle(H)), color=ORANGE, lw=1.8)
    a2.axvline(fc, color=DOT, lw=1.0)
    a2.axhline(-45 if lo else 45, color=MUTED, lw=0.8, ls=":")
    a2.axvline(f_hz, color=FG, lw=1.0, ls="--")
    a2.set_ylim(-100, 100); a2.set_xlabel("frequency  (Hz)")
    a2.set_ylabel("phase  (deg)")
    a2.set_title(f"phase passes {'−45' if lo else '+45'}° exactly at the corner")

    i10 = np.argmin(np.abs(ff - fc * 10)); i100 = np.argmin(np.abs(ff - fc * 100))
    slope = 20 * np.log10(abs(H[i100])) - 20 * np.log10(abs(H[i10]))
    readout(fig, 0.845, 0.90, [
        "FILTER", "─" * 26,
        f"{kind.split()[0]:>26s}",
        f"R           {R:>10.0f}Ω",
        f"C           {C_nF:>10.1f}nF",
        f"fc = 1/2πRC {fc:>10.2f}Hz",
        f"τ = RC      {R*C*1e6:>10.3f}µs",
        "", "AT THIS f", "─" * 26,
        f"f           {f_hz:>10.1f}Hz",
        f"f/fc        {f_hz/fc:>10.4f}",
        f"|H|         {abs(Hn):>10.5f}",
        f"|H| dB      {20*np.log10(abs(Hn)):>+10.3f}dB",
        f"phase       {np.degrees(np.angle(Hn)):>+10.3f}°",
        "", "AT THE CORNER", "─" * 26,
        f"|H|         {1/np.sqrt(2):>10.5f}",
        f"dB          {-3.0103:>+10.4f}dB",
        f"power ratio {0.5:>10.4f}",
        "", "ASYMPTOTE", "─" * 26,
        f"10fc→100fc  {slope:>+10.3f}dB",
        "one pole = 20 dB/decade",
        "         = 90° of phase",
    ])
    footer(fig, f"H = 1/(1 + jf/fc)   ·   fc = {fc:,.1f} Hz   ·   "
                f"−3.01 dB and −45° at the corner   ·   slope {slope:.1f} dB/decade")
    plt.show()


w6 = dict(f_hz=widgets.FloatSlider(value=1592, min=10, max=200000, step=10,
                                   description="frequency Hz:", **SL),
          R=widgets.FloatSlider(value=1000, min=100, max=10000, step=100,
                                description="R (Ω):", **SL),
          C_nF=widgets.FloatSlider(value=100, min=1, max=1000, step=1,
                                   description="C (nF):", **SL),
          kind=widgets.Dropdown(options=["low-pass  (C to ground)",
                                         "high-pass  (R to ground)"],
                                value="low-pass  (C to ground)",
                                description="filter:", **SL))
display(widgets.HBox([w6["f_hz"], w6["R"], w6["C_nF"], w6["kind"]]),
        widgets.interactive_output(draw_bode, w6))

Output()